# KD surveillance echocardiography — v5 temporal validation (2025–2026)
Wuhan Children's Hospital · Kawasaki disease research kit v1 + v1.1 addendum · American English · every number is computed.

**Question.** Do the locked v4 estimates of the yield of post-discharge echocardiography after an uncomplicated in-hospital course hold in later children whose echocardiograms carry structured coronary measurements?

**Plan.** `V5_VALIDATION_PLAN.txt`, locked on 14 October 2026 (10:47 UTC) before any validation outcome was computed; SHA-256 `8d6b811b2d42ac80cdc4438b7adbeb9623e39ff962899e710c72c0c6f482065f` (checked in section 1). The locked v4 and v4.1 results are not changed.

* **Data.** Structured echocardiography export, 2025-03-08 to 2026-09-23 (7,213 reports, 2,596 children), frozen as the kit v1.1 tables `KD_echo_measure_2025_2026_v1` and `KD_echo_reports_combined_v5`.
* **Per-report category.** AHA 2017 categories from the maximal Z of the structured proximal values plus the numbers written in the report text, with a wording override for non-proximal segments; tier 3/4 fallback as in the kit.
* **Cohort V.** Index admissions whose first in-hospital echocardiogram is on or after 2025-03-08.
* **Primary endpoint P1.** Aneurysm at the first classifiable echocardiogram 1–30 days after discharge among uncomplicated children with complete 30-day follow-up.
* **Comparison with v4 (26/1,618).** Newcombe difference, beta-binomial predictive compatibility (P ≥ 0.05 = replicated) and a pooled 2018–2026 estimate.
* **Secondary endpoints S1–S8** (plan section 6) and two sensitivity analyses.

**Outputs** (`output_kd_echo_surveillance_v5/`):
* `Results_locked_v5.txt` and `key_numbers_v5.json`;
* Tables V5-1 to V5-4 and Figure V5-1 (legends in `Figures_and_tables_index_v5.txt`);
* `CHECKSUMS_v5.sha256`;
* the kit v1.1 manifest in `KD_ResearchKit_v1_1_addendum/MANIFEST_v1_1.csv`.

Data are pseudonymized (research IDs only) and for internal team use. Ethics approval: Ethics Committee of Wuhan Children's Hospital, 2024R034-E02.

## 1. Setup
Place this notebook in a folder that also holds the unzipped `KD_ResearchKit_v1/`, `KD_ResearchKit_v1_1_addendum/`, `V5_VALIDATION_PLAN.txt` and, if available, the structured echocardiography export (`.xlsx` with the columns `lca`, `lca_z`, ...). Outputs go to `output_kd_echo_surveillance_v5/`.

* The plan file must be byte-identical to the version locked on 14 October 2026 (checked by SHA-256).
* With the export present, the two kit v1.1 tables are rebuilt from it and checked against the frozen files (the frozen files are written only if they do not exist yet). Without the export, the frozen tables are used.

In [ ]:
import os, re, sys, json, glob, hashlib, datetime
import numpy as np, pandas as pd
from scipy import stats
BASE = os.path.abspath('.')
KIT = os.path.join(BASE, 'KD_ResearchKit_v1')
sys.path.insert(0, os.path.join(KIT, 'code'))
import kdkit
OUT = os.path.join(BASE, 'output_kd_echo_surveillance_v5'); os.makedirs(OUT, exist_ok=True)
ADD = os.path.join(BASE, 'KD_ResearchKit_v1_1_addendum'); os.makedirs(ADD, exist_ok=True)
kit = kdkit.load(KIT)
FD = os.path.join(KIT, 'data_frozen'); RL = os.path.join(KIT, 'results_locked')


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def find_export():
    need = {'科研就诊编号', '科研患者编号', '检查时间', 'lca', 'lca_z', 'lad', 'lad_z', 'rcaj', 'rcaj_z'}
    for p in sorted(glob.glob(os.path.join(BASE, '*.xlsx'))):
        try:
            cols = set(pd.read_excel(p, nrows=0).columns)
        except Exception:
            continue
        if need.issubset(cols):
            return p
    return None


def freeze(df, path):
    """Write a frozen table once. If it already exists, check that the rebuilt table is identical and leave the file untouched."""
    if os.path.exists(path):
        old = pd.read_parquet(path)
        pd.testing.assert_frame_equal(old.reset_index(drop=True), df.reset_index(drop=True), check_dtype=False)
        print(f'{os.path.basename(path)}: already frozen; rebuilt table identical (file not rewritten)')
    else:
        df.to_parquet(path, index=False, compression='zstd', compression_level=19)
        df.to_csv(path.replace('.parquet', '.csv'), index=False, encoding='utf-8-sig')
        print(f'{os.path.basename(path)}: frozen now')
    print('  SHA-256', sha256(path))


PLAN = os.path.join(BASE, 'V5_VALIDATION_PLAN.txt')
PLAN_SHA256_LOCKED = '8d6b811b2d42ac80cdc4438b7adbeb9623e39ff962899e710c72c0c6f482065f'   # recorded when the plan was locked, 2026-10-14T10:47:36Z
assert os.path.exists(PLAN) and sha256(PLAN) == PLAN_SHA256_LOCKED, 'V5_VALIDATION_PLAN.txt is missing or differs from the locked version'
print('validation plan: identical to the locked version, SHA-256', PLAN_SHA256_LOCKED)
EXPORT = find_export()
EXPORT_SHA256 = sha256(EXPORT) if EXPORT else None
print('export file:', f'{os.path.basename(EXPORT)} | SHA-256 {EXPORT_SHA256}' if EXPORT else 'not found next to the notebook; the frozen kit v1.1 measurement table is used')

## 2. Freeze the structured measurement table (kit v1.1)
One row per exported report. Cells that are not plain numbers are parsed by taking the largest well-formed number (for example `3.3-6.9` gives 6.9 and `6.4近段` gives 6.4); malformed cells (for example `-0.3.8`) become missing and are flagged. Z values are recomputed with the laboratory equations of the kit (Stevenson body surface area) to check that the export follows the same equations.

In [ ]:
RAW_COLS = {'科研就诊编号': 'encounter_id', '科研患者编号': 'research_patient_id', '检查时间': 'exam_time',
            '儿童超声心动图所见': 'findings', '儿童超声心动图结论': 'conclusion', '射血分数': 'ejection_fraction',
            '缩短率': 'fractional_shortening', 'lca': 'lca_raw', 'lad': 'lad_raw', 'rcaj': 'rca_raw',
            'lca_z': 'lca_z_raw', 'lad_z': 'lad_z_raw', 'rcaj_z': 'rca_z_raw', 'lcx': 'lcx_raw',
            'height': 'height_cm', 'weight': 'weight_kg'}
NUMRE = re.compile(r'(?<![\d.])-?\d+(?:\.\d+)?(?![\d.])')


def parse_max(v):
    if pd.isna(v):
        return np.nan, ''
    s = str(v).strip()
    try:
        return float(s), ''
    except ValueError:
        pass
    nums = [float(n) for n in NUMRE.findall(s)]
    return (max(nums), 'parsed_max') if nums else (np.nan, 'non_numeric')


def build_measure_table(path):
    raw = pd.read_excel(path, dtype=object).rename(columns=RAW_COLS)
    M = pd.DataFrame({'research_patient_id': raw['research_patient_id'].astype(str), 'encounter_id': raw['encounter_id'].astype(str),
                      'exam_date': pd.to_datetime(raw['exam_time']).dt.normalize()})
    flags = []
    for src, dst in [('lca_raw', 'lca_mm'), ('lad_raw', 'lad_mm'), ('rca_raw', 'rca_prox_mm'), ('lca_z_raw', 'lca_z'), ('lad_z_raw', 'lad_z'), ('rca_z_raw', 'rca_prox_z')]:
        parsed = raw[src].map(parse_max)
        M[dst] = parsed.map(lambda t: t[0]); flags.append(parsed.map(lambda t, dst=dst: f'{dst}:{t[1]}' if t[1] else ''))
    M['parse_flags'] = pd.concat(flags, axis=1).apply(lambda r: ';'.join([f for f in r if f]), axis=1)
    LCX_TEXT = {'开口可见': 'origin_visible', '显示受限': 'limited_view', '可显示': 'visible', '可见': 'visible'}
    M['lcx_mm'] = pd.to_numeric(raw['lcx_raw'], errors='coerce')
    M['lcx_note'] = raw['lcx_raw'].map(lambda v: '' if pd.isna(v) else LCX_TEXT.get(str(v).strip(), 'number' if pd.notna(pd.to_numeric(v, errors='coerce')) else 'other'))
    for c in ['height_cm', 'weight_kg', 'ejection_fraction', 'fractional_shortening']:
        M[c] = pd.to_numeric(raw[c], errors='coerce')
    M['bsa_stevenson'] = kdkit.stevenson_bsa(M['height_cm'], M['weight_kg'])
    EQ = pd.read_csv(os.path.join(KIT, 'reference', 'lab_Z_equations_full_precision.csv')).set_index('vessel')
    for v, d, z in [('LMCA', 'lca_mm', 'lca_z'), ('LAD', 'lad_mm', 'lad_z'), ('RCA', 'rca_prox_mm', 'rca_prox_z')]:
        r = EQ.loc[v]
        M[z + '_recomputed'] = (M[d] - (r['a'] + r['b_sqrtBSA'] * np.sqrt(M['bsa_stevenson']) + r['c_BSA'] * M['bsa_stevenson'])) / r['SD_mm']
    diffs = pd.concat([(M[z + '_recomputed'] - M[z]).abs() for z in ['lca_z', 'lad_z', 'rca_prox_z']], axis=1)
    M['z_matches_lab_equations'] = np.where(diffs.notna().any(axis=1), (diffs.fillna(0) <= 0.05).all(axis=1), np.nan)
    M['n_rows_same_day'] = M.groupby(['research_patient_id', 'exam_date'])['exam_date'].transform('size')
    return M.sort_values(['research_patient_id', 'exam_date', 'encounter_id'], kind='mergesort').reset_index(drop=True)


p_meas = os.path.join(ADD, 'KD_echo_measure_2025_2026_v1.parquet')
if EXPORT:
    MEAS = build_measure_table(EXPORT)
    freeze(MEAS, p_meas)
else:
    MEAS = pd.read_parquet(p_meas)
    print(f'{os.path.basename(p_meas)} loaded | SHA-256', sha256(p_meas))
print(f"measurement table: {len(MEAS)} reports, {MEAS['research_patient_id'].nunique()} children, "
      f"{MEAS['exam_date'].min().date()} to {MEAS['exam_date'].max().date()}")
for c in ['lca_mm', 'lad_mm', 'rca_prox_mm', 'lca_z', 'lad_z', 'rca_prox_z', 'height_cm']:
    print(f'  {c}: numeric {MEAS[c].notna().sum()} ({100 * MEAS[c].notna().mean():.1f}%)')
for c in ['lca_z', 'lad_z', 'rca_prox_z']:
    d_ = (MEAS[c + '_recomputed'] - MEAS[c]).abs().dropna()
    print(f'  {c}: recomputed with the laboratory equations, median |difference| {d_.median():.4f}; within 0.05 in {100 * (d_ <= 0.05).mean():.1f}%')
EXPORT_START, DATA_END = MEAS['exam_date'].min(), MEAS['exam_date'].max()

## 3. Per-report combined coronary category (kit v1.1)
For every frozen report dated inside the export window, the category is taken from all available numbers: the structured proximal values plus any values written in the report text (evidence tiers 1–2 of the kit), with diameters of 8 mm or more counted as large/giant. The structured table holds proximal segments only, so a report without text numbers whose wording-model category is an aneurysm and whose conclusion names a non-proximal segment keeps the wording-model category (wording override). Reports without any numbers fall back to transcribed numbers (tier 3) and then to the wording model (tier 4), as in the kit.

In [ ]:
E = kit['KD_echo_reports_AHA_v1'].copy()
n_dup_keys = int(E['report_key'].duplicated().sum())
E = E.drop_duplicates('report_key', keep='first')   # exact duplicate rows of the same report in the frozen export
print('duplicate report rows dropped from the frozen report table:', n_dup_keys)
E['research_patient_id'] = E['research_patient_id'].astype(str); E['exam_date'] = pd.to_datetime(E['exam_time']).dt.normalize()
TXT = kit['KD_echo_reports_text_v1'][['report_key', 'conclusion']].drop_duplicates('report_key', keep='first')
EW = E[(E['exam_date'] >= EXPORT_START) & (E['exam_date'] <= DATA_END)].merge(TXT, on='report_key', how='left')
DAY = MEAS.groupby(['research_patient_id', 'exam_date']).agg(
    table_zmax=('lca_z', 'max'), lad_zmax=('lad_z', 'max'), rca_zmax=('rca_prox_z', 'max'),
    table_dmax=('lca_mm', 'max'), lad_dmax=('lad_mm', 'max'), rca_dmax=('rca_prox_mm', 'max'), n_table_rows=('exam_date', 'size')).reset_index()
DAY['table_zmax'] = DAY[['table_zmax', 'lad_zmax', 'rca_zmax']].max(axis=1)
DAY['table_dmax'] = DAY[['table_dmax', 'lad_dmax', 'rca_dmax']].max(axis=1)
EW = EW.merge(DAY[['research_patient_id', 'exam_date', 'table_zmax', 'table_dmax', 'n_table_rows']], on=['research_patient_id', 'exam_date'], how='left')
text_num = EW['tier'].isin([1, 2])
EW['text_zmax'] = np.where(text_num, EW['maxZ'], np.nan); EW['text_dmax'] = np.where(text_num, EW['maxD'], np.nan)
num_z = EW[['table_zmax', 'text_zmax']].max(axis=1); num_d = EW[['table_dmax', 'text_dmax']].max(axis=1)
EW['num_cat'] = np.where(num_z.notna() | (num_d >= 8), kdkit.aha_category(num_z.values, num_d.values), np.nan)
EW['wording_cat'] = np.where(EW['tier'] == 4, EW['aha_best'], np.nan)
NONPROX = re.compile(r'中段|远段|中远|远端|分叉|回旋支|LCX|后降支|左室后支|后侧支')
conc = EW['conclusion'].fillna('').astype(str)
EW['nonproximal_wording'] = conc.str.contains(NONPROX) & conc.str.contains(r'扩张|瘤')
override = EW['num_cat'].notna() & (EW['num_cat'] < 2) & ~text_num & (EW['wording_cat'] >= 2) & EW['nonproximal_wording']
fallback_t3 = EW['num_cat'].isna() & (EW['tier'] == 3)
fallback_t4 = EW['num_cat'].isna() & (EW['tier'] == 4)
EW['combined_cat'] = np.select([override, EW['num_cat'].notna(), fallback_t3, fallback_t4],
                               [EW['wording_cat'], EW['num_cat'], EW['aha_best'], EW['wording_cat']], np.nan)
EW['combined_src'] = np.select([override, EW['num_cat'].notna(), fallback_t3, fallback_t4],
                               ['numbers+wording_override', 'numbers', 'tier3_transcribed', 'wording_only'], 'none')
EW['cat_no_override'] = np.select([EW['num_cat'].notna(), fallback_t3, fallback_t4], [EW['num_cat'], EW['aha_best'], EW['wording_cat']], np.nan)
EW['cat_wording_v4'] = EW['aha_best']
EW['multi_report_same_day'] = EW.groupby(['research_patient_id', 'exam_date'])['report_key'].transform('size') > 1
COMB = EW[['report_key', 'research_patient_id', 'encounter_id', 'exam_date', 'phase', 'tier', 'cat_wording_v4', 'table_zmax', 'table_dmax',
           'text_zmax', 'text_dmax', 'num_cat', 'wording_cat', 'nonproximal_wording', 'combined_cat', 'combined_src', 'cat_no_override',
           'n_table_rows', 'multi_report_same_day']].sort_values(['research_patient_id', 'exam_date', 'report_key'], kind='mergesort').reset_index(drop=True)
p_comb = os.path.join(ADD, 'KD_echo_reports_combined_v5.parquet')
freeze(COMB, p_comb)
print(f'frozen reports in the export window: {len(COMB)}; matched to the structured table: {int(COMB["n_table_rows"].notna().sum())}')
print('source of the combined category:'); print(COMB['combined_src'].value_counts().to_string())
print('\ncombined category vs v4 wording/numeric category (all reports in the window):')
print(pd.crosstab(COMB['cat_wording_v4'], COMB['combined_cat'], rownames=['v4 category'], colnames=['combined'], dropna=False).to_string())

## 4. Locked v4 reference values
Recomputed from the frozen cohort table `cohort_C1_primary_v4` and checked against `Results_locked_v4.txt`.

In [ ]:
C1 = kit['cohort_C1_primary_v4'].copy()
C1['research_patient_id'] = C1['科研患者编号'].astype(str)
coh4 = C1[(C1['in_max'] == 0) & (C1['post30_n'] > 0)]
V4 = {'P1': (int((coh4['post30_cat_first'] >= 2).sum()), len(coh4)),
      'S1': (int((coh4['post30_cat_first'] >= 1).sum()), len(coh4)),
      'S2': (int((coh4['post30_cat_first'] >= 3).sum()), len(coh4))}
w2_4 = coh4[(coh4['post30_cat_max'] == 0) & (coh4['d31_90_cat_max'].notna())]
V4['S4_aneurysm'] = (int((w2_4['d31_90_cat_max'] >= 2).sum()), len(w2_4))
V4['S4_involvement'] = (int((w2_4['d31_90_cat_max'] >= 1).sum()), len(w2_4))
LOCK4_PATH = os.path.join(RL, 'output_kd_echo_surveillance_v4', 'Results_locked_v4.txt')
LOCK4 = open(LOCK4_PATH, encoding='utf-8').read()
print('Results_locked_v4.txt SHA-256', sha256(LOCK4_PATH))
for k, (a, n) in V4.items():
    s = f'{a:,}/{n:,}'
    found = re.search(r'(?<![\d,.])' + re.escape(s) + r'(?![\d,])', LOCK4) is not None
    print(f'v4 {k}: {s}', '| found in Results_locked_v4.txt' if found else '| NOT FOUND in the lock file')
    assert found

## 5. Validation cohort (V)
Index admissions whose first in-hospital echocardiogram is on or after the first exam date of the structured export; per-report categories from the combined rule (plan section 4).

In [ ]:
MI = kit['KD_master_index_v1'].copy()
MI['research_patient_id'] = MI['research_patient_id'].astype(str)
MI['dis'] = pd.to_datetime(MI['index_discharge_date']).dt.normalize()
first_inh = E[E['phase'] == 'in_index'].groupby('research_patient_id')['exam_date'].min()
MI['first_inh'] = MI['research_patient_id'].map(first_inh)
CAND = MI[MI['first_inh'] >= EXPORT_START].set_index('research_patient_id')


def day_max(df, cat):
    d = df[df[cat].notna()]
    return d.groupby(['research_patient_id', 'exam_date'])[cat].max().reset_index()


def build(cat, frame=None):
    """Child-level windows for the validation cohort using the per-report category column `cat` (of COMB unless another frame is given)."""
    F = COMB if frame is None else frame
    R = F[F['research_patient_id'].isin(CAND.index)]
    dm = {ph: day_max(R[R['phase'] == ph], cat) for ph in ['in_index', 'post30', 'd31_90', 'd91_365', 'gt365']}
    ch = pd.DataFrame(index=CAND.index)
    ch['dis'] = CAND['dis']
    ch['inmax'] = dm['in_index'].groupby('research_patient_id')[cat].max()
    p30 = dm['post30'].sort_values(['research_patient_id', 'exam_date'])
    ch['q1_cat'] = p30.groupby('research_patient_id')[cat].first()
    ch['q1_date'] = p30.groupby('research_patient_id')['exam_date'].first()
    ch['p30_max'] = p30.groupby('research_patient_id')[cat].max()
    for ph, nm in [('d31_90', 'w2_max'), ('d91_365', 'w3_max'), ('gt365', 'w4_max')]:
        ch[nm] = dm[ph].groupby('research_patient_id')[cat].max()
    return ch, dm


CH, DM = build('combined_cat')
V = CH[CH['inmax'].notna()]
print(f'index admissions with all in-hospital echocardiograms in the structured period: {len(CAND)}; with a classifiable in-hospital report (cohort V): {len(V)}')
print('index year of V:', CAND.loc[V.index, 'index_year'].value_counts().sort_index().to_dict())
print('in-hospital maximum category in V:', V['inmax'].value_counts().sort_index().to_dict())
U = V[V['inmax'] == 0]
END30, END90, END365 = DATA_END - pd.Timedelta(days=30), DATA_END - pd.Timedelta(days=90), DATA_END - pd.Timedelta(days=365)
U30 = U[U['dis'] <= END30]
Q1 = U30[U30['q1_cat'].notna()]
print(f'uncomplicated: {len(U)}; with complete 30-day follow-up: {len(U30)}; with a first echocardiogram 1-30 days after discharge: {len(Q1)}')

## 6. Primary and secondary endpoints

In [ ]:
def rate(k, n):
    lo, hi = kdkit.wilson(k, n)
    return dict(events=int(k), n=int(n), pct=round(100 * k / n, 2) if n else np.nan, ci_low=round(lo, 2), ci_high=round(hi, 2))


def newcombe_diff(k1, n1, k2, n2):
    """Difference p1 - p2 (percentage points) with the Newcombe hybrid score 95% CI."""
    p1, p2 = k1 / n1, k2 / n2
    l1, u1 = [v / 100 for v in kdkit.wilson(k1, n1)]
    l2, u2 = [v / 100 for v in kdkit.wilson(k2, n2)]
    d = p1 - p2
    lo = d - np.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2)
    hi = d + np.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)
    return round(100 * d, 2), round(100 * lo, 2), round(100 * hi, 2)


def predictive_p(x, n, k_ref, n_ref, digits=4):
    """Two-sided beta-binomial predictive P for x events in n given k_ref/n_ref, prior Beta(k_ref + 0.5, n_ref - k_ref + 0.5)."""
    a, b = k_ref + 0.5, n_ref - k_ref + 0.5
    lo = stats.betabinom.cdf(x, n, a, b); hi = stats.betabinom.sf(x - 1, n, a, b)
    p = float(min(1.0, 2 * min(lo, hi)))
    return round(p, digits) if digits is not None else p


def compare(name, k, n, ref):
    kr, nr = ref
    d, lo, hi = newcombe_diff(k, n, kr, nr)
    pp = predictive_p(k, n, kr, nr)
    pooled = rate(k + kr, n + nr)
    return dict(endpoint=name, **{f'v5_{key}': val for key, val in rate(k, n).items()},
                **{f'v4_{key}': val for key, val in rate(kr, nr).items()},
                diff_pp=d, diff_ci_low=lo, diff_ci_high=hi, predictive_P=pp,
                reading=('replicated' if pp >= 0.05 else 'not replicated'),
                pooled_events=pooled['events'], pooled_n=pooled['n'], pooled_pct=pooled['pct'], pooled_ci_low=pooled['ci_low'], pooled_ci_high=pooled['ci_high'])


ROWS = []
ROWS.append(dict(key='P1', prespecified_comparison=True, **compare('P1 aneurysm at the first echocardiogram 1-30 d', int((Q1['q1_cat'] >= 2).sum()), len(Q1), V4['P1'])))
ROWS.append(dict(key='S1', prespecified_comparison=True, **compare('S1 any involvement at the first echocardiogram 1-30 d', int((Q1['q1_cat'] >= 1).sum()), len(Q1), V4['S1'])))
ROWS.append(dict(key='S2', prespecified_comparison=True, **compare('S2 medium or larger at the first echocardiogram 1-30 d', int((Q1['q1_cat'] >= 3).sum()), len(Q1), V4['S2'])))
S3 = rate(int((Q1['p30_max'] >= 2).sum()), len(Q1))
W2 = U[(U['dis'] <= END90) & (U['p30_max'] == 0) & U['w2_max'].notna()]
ROWS.append(dict(key='S4_aneurysm', prespecified_comparison=True, **compare('S4 aneurysm at 31-90 d after normal 1-30-d echocardiograms', int((W2['w2_max'] >= 2).sum()), len(W2), V4['S4_aneurysm'])))
ROWS.append(dict(key='S4_involvement', prespecified_comparison=False, **compare('S4 any involvement at 31-90 d after normal 1-30-d echocardiograms (comparison not pre-specified)', int((W2['w2_max'] >= 1).sum()), len(W2), V4['S4_involvement'])))
W3 = U[(U['dis'] <= END365) & (U['p30_max'] == 0) & (U['w2_max'] == 0) & U['w3_max'].notna()]
S5 = {'aneurysm': rate(int((W3['w3_max'] >= 2).sum()), len(W3)), 'involvement': rate(int((W3['w3_max'] >= 1).sum()), len(W3))}
CMP = pd.DataFrame(ROWS)
for k in ['P1', 'S1', 'S2', 'S4_aneurysm', 'S4_involvement']:   # expected count in V under the v4 proportion (beta-binomial mean)
    r_ = CMP['key'] == k; kr, nr = (V4['S4_aneurysm'] if k == 'S4_aneurysm' else V4['S4_involvement'] if k == 'S4_involvement' else V4[k])
    CMP.loc[r_, 'expected_v5_events_under_v4'] = round(float(CMP.loc[r_, 'v5_n'].iloc[0]) * (kr + 0.5) / (nr + 1), 2)
CMP.to_csv(os.path.join(OUT, 'TableV5_1_validation_vs_v4.csv'), index=False)
pd.set_option('display.width', 250)
print(CMP[['endpoint', 'v5_events', 'v5_n', 'v5_pct', 'v5_ci_low', 'v5_ci_high', 'v4_events', 'v4_n', 'v4_pct', 'diff_pp', 'diff_ci_low', 'diff_ci_high', 'expected_v5_events_under_v4', 'predictive_P', 'reading', 'pooled_pct', 'pooled_ci_low', 'pooled_ci_high']].to_string(index=False))
print('S3 aneurysm on any echocardiogram within 30 days:', S3)
print('S5 91-365 d after normal findings in all earlier windows:', S5)

## 7. Persistence of the new aneurysms (S6) and the risk-factor strategy (S7)

In [ ]:
ALLD = day_max(COMB[COMB['research_patient_id'].isin(CAND.index)], 'combined_cat')
ev = Q1[Q1['q1_cat'] >= 2]
pers, gap = [], []
for pid, r in ev.iterrows():
    nxt = ALLD[(ALLD['research_patient_id'] == pid) & (ALLD['exam_date'] > r['q1_date']) & (ALLD['exam_date'] <= r['q1_date'] + pd.Timedelta(days=90))].sort_values('exam_date')
    pers.append('not re-imaged' if nxt.empty else ('confirmed' if nxt['combined_cat'].iloc[0] >= 2 else 'not confirmed'))
    gap.append(np.nan if nxt.empty else (nxt['exam_date'].iloc[0] - r['q1_date']).days)
ev = ev.assign(next_90d=pers, days_to_next=gap)
S6 = ev['next_90d'].value_counts().reindex(['confirmed', 'not confirmed', 'not re-imaged']).fillna(0).astype(int).to_dict()
S6_size = ev['q1_cat'].map({2: 'small', 3: 'medium', 4: 'large/giant'}).value_counts().to_dict()
S6_gap = ev['days_to_next'].dropna()
print('S6 persistence of P1 events (next classifiable echocardiogram within 90 days):', S6, '| size at detection:', S6_size,
      f'| days to the next echocardiogram: median {S6_gap.median():.1f} (range {S6_gap.min():.0f}-{S6_gap.max():.0f})' if len(S6_gap) else '')
# S7: risk-factor strategy in children discharged with in-hospital maximum <= 1
D25 = V[(V['inmax'] <= 1) & (V['dis'] <= END30) & V['q1_cat'].notna()].copy()
X = CAND.loc[D25.index]
D25['rf_age'] = X['age_months'] < 12
D25['rf_ivigr'] = X['ivig_resistant_orders'] == 1
D25['rf_dil'] = D25['inmax'] == 1
D25['rf_infl'] = (X['lab_CRPC_dis'] > 20) | (X['lab_WBC_dis'] > 12)
D25['n_rf'] = D25[['rf_age', 'rf_ivigr', 'rf_dil', 'rf_infl']].sum(axis=1)
D25['an'] = D25['q1_cat'] >= 2
S7 = dict(children=len(D25), aneurysms=int(D25['an'].sum()), echoes_if_ge1=int((D25['n_rf'] >= 1).sum()),
          echoes_avoided_pct=round(100 * (D25['n_rf'] == 0).mean(), 1), detected=int((D25['an'] & (D25['n_rf'] >= 1)).sum()),
          missed=int((D25['an'] & (D25['n_rf'] == 0)).sum()),
          missed_sizes=D25.loc[D25['an'] & (D25['n_rf'] == 0), 'q1_cat'].map({2: 'small', 3: 'medium', 4: 'large/giant'}).value_counts().to_dict(),
          by_count={int(k): f"{int(g['an'].sum())}/{len(g)}" for k, g in D25.groupby(D25['n_rf'].clip(upper=2))})
S7['sensitivity'] = rate(S7['detected'], S7['aneurysms']) if S7['aneurysms'] else None
print('S7 risk-factor strategy (>=1 factor):', S7)

## 8. Sensitivity analyses of P1 (measurement source)

In [ ]:
SENS = []
for cat, lab in [('combined_cat', 'Primary: combined rule'), ('cat_no_override', '(a) without the wording override'), ('cat_wording_v4', '(b) v4 report categories (wording model in this period)')]:
    ch_, _ = build(cat)
    v_ = ch_[ch_['inmax'].notna()]; u_ = v_[(v_['inmax'] == 0) & (v_['dis'] <= END30)]; q_ = u_[u_['q1_cat'].notna()]
    SENS.append(dict(analysis=lab, **rate(int((q_['q1_cat'] >= 2).sum()), len(q_)), predictive_P=predictive_p(int((q_['q1_cat'] >= 2).sum()), len(q_), *V4['P1'])))
SENS = pd.DataFrame(SENS); SENS.to_csv(os.path.join(OUT, 'TableV5_2_P1_sensitivity.csv'), index=False)
print(SENS.to_string(index=False))

## 9. Extended follow-up of the locked v4 cohort C1 (S8)
Later windows recomputed with echocardiograms through the end of the structured export: kit categories before the structured period, combined categories inside it. First the locked v4 denominators are reproduced with echocardiograms through 28 February 2025.

In [ ]:
c1ids = set(C1['research_patient_id'])
EC = E[E['research_patient_id'].isin(c1ids)][['report_key', 'research_patient_id', 'exam_date', 'phase', 'aha_best']].copy()
EC = EC.merge(COMB[['report_key', 'combined_cat']], on='report_key', how='left')


def windows(cat_col, end):
    d = EC[(EC['exam_date'] <= end) & EC[cat_col].notna()]
    out = pd.DataFrame(index=sorted(c1ids))
    for ph in ['post30', 'd31_90', 'd91_365', 'gt365']:
        out[ph] = d[d['phase'] == ph].groupby('research_patient_id')[cat_col].max()
    out['in_max'] = C1.set_index('research_patient_id')['in_max']
    return out


EC['cat_v4'] = EC['aha_best']
w_lock = windows('cat_v4', pd.Timestamp('2025-02-28'))
EC['cat_ext'] = np.where(EC['exam_date'] >= EXPORT_START, EC['combined_cat'], EC['aha_best'])
w_ext = windows('cat_ext', DATA_END)
S8 = []
for lab, w in [('locked v4 scope (echocardiograms to 2025-02-28)', w_lock), ('extended to the end of the structured export', w_ext)]:
    u = w[w['in_max'] == 0]
    s3 = u[(u['post30'] == 0) & (u['d31_90'] == 0) & u['d91_365'].notna()]
    s4 = u[(u['post30'] == 0) & (u['d31_90'] == 0) & (u['d91_365'] == 0) & u['gt365'].notna()]
    S8.append(dict(scope=lab, W3=f"{int((s3['d91_365'] >= 2).sum())}/{len(s3)}", W3_involvement=f"{int((s3['d91_365'] >= 1).sum())}/{len(s3)}",
                   W4=f"{int((s4['gt365'] >= 2).sum())}/{len(s4)}", W4_involvement=f"{int((s4['gt365'] >= 1).sum())}/{len(s4)}",
                   W3_ci='%.2f–%.2f' % kdkit.wilson(int((s3['d91_365'] >= 2).sum()), len(s3)), W4_ci='%.2f–%.2f' % kdkit.wilson(int((s4['gt365'] >= 2).sum()), len(s4))))
S8 = pd.DataFrame(S8); S8.to_csv(os.path.join(OUT, 'TableV5_3_extended_followup_C1.csv'), index=False)
print(S8.to_string(index=False))

## 10. Additional descriptive analyses (not pre-specified)
Reported as additions in the results lock: (a) the P1 events one by one; (b) confirmation of the new aneurysms compared with the same definition in the locked v4.1 results (module M2b, definition a); (c) timing and uptake of the first post-discharge echocardiogram; (d) the v4 value of S3 for context.

In [ ]:
CATNAME = {0: 'none', 1: 'dilation only', 2: 'small', 3: 'medium', 4: 'large/giant'}
RV = COMB[COMB['research_patient_id'].isin(V.index)]
det = []
for pid, r in ev.sort_values('q1_date').iterrows():
    rr = RV[RV['research_patient_id'] == pid]
    inh = rr[rr['phase'] == 'in_index']; day = rr[rr['exam_date'] == r['q1_date']]
    later = ALLD[(ALLD['research_patient_id'] == pid) & (ALLD['exam_date'] > r['q1_date'])].sort_values('exam_date')
    det.append(dict(research_patient_id=pid, index_year=int(CAND.loc[pid, 'index_year']), inhospital_report_days=int(inh['exam_date'].nunique()),
                    inhospital_max_structured_Z=round(float(inh['table_zmax'].max()), 2), day_after_discharge=int((r['q1_date'] - r['dis']).days),
                    category=CATNAME[int(r['q1_cat'])], structured_Z=round(float(day['table_zmax'].max()), 2),
                    text_Z=round(float(day['text_zmax'].max()), 2) if day['text_zmax'].notna().any() else np.nan,
                    source=';'.join(sorted(day['combined_src'].unique())), status_next_90d=r['next_90d'],
                    next_echo_days_later=int((later['exam_date'].iloc[0] - r['q1_date']).days) if len(later) else np.nan,
                    next_echo_category=CATNAME[int(later['combined_cat'].iloc[0])] if len(later) else '',
                    last_echo_day_after_discharge=int((later['exam_date'].iloc[-1] - r['dis']).days) if len(later) else np.nan,
                    last_echo_category=CATNAME[int(later['combined_cat'].iloc[-1])] if len(later) else '', later_echo_days=len(later)))
DET = pd.DataFrame(det); DET.to_csv(os.path.join(OUT, 'TableV5_4_P1_events.csv'), index=False)
print(DET.to_string(index=False))

# (b) confirmation compared with v4.1 module M2b, definition (a)
REF41 = os.path.join(BASE, 'reference_locked', 'Results_locked_v4_1.txt')
REF41_SHA256 = '5600d5765bb34348dcfeab220325a4a7abe1e8b28a979d1ee6ac18777d14ee55'
if os.path.exists(REF41):
    assert sha256(REF41) == REF41_SHA256, 'reference_locked/Results_locked_v4_1.txt differs from the locked v4.1 file'
    m_ = re.search(r'\(a\) persistence: Z ≥ 2\.5 again on the next echo ≤90 d later: (\d+)/([\d,]+) = .*?\(not confirmed (\d+), not re-imaged ≤90 d (\d+)\)', open(REF41, encoding='utf-8').read())
    c41, n41, nc41, nr41 = int(m_.group(1)), int(m_.group(2).replace(',', '')), int(m_.group(3)), int(m_.group(4))
    print('v4.1 M2b(a) read from the locked file:', c41, n41, nc41, nr41)
else:
    c41, n41, nc41, nr41 = 13, 1618, 9, 4   # values printed in Results_locked_v4_1.txt (SHA-256 above), module M2b, definition (a)
    print('reference_locked/Results_locked_v4_1.txt not found; the recorded v4.1 values are used')
conf5, nc5, nr5 = S6['confirmed'], S6['not confirmed'], S6['not re-imaged']
ADDL = dict(confirmed_yield_v5=rate(conf5, len(Q1)), confirmed_yield_v41=rate(c41, n41),
            confirmed_among_reimaged_v5=rate(conf5, conf5 + nc5), confirmed_among_reimaged_v41=rate(c41, c41 + nc41),
            not_reimaged_v5=nr5, not_reimaged_v41=nr41,
            fisher_P_confirmed_among_reimaged_posthoc=round(float(stats.fisher_exact([[conf5, nc5], [c41, nc41]])[1]), 4))
# (c) timing and uptake of the first post-discharge echocardiogram
q1day = (Q1['q1_date'] - Q1['dis']).dt.days
u4 = C1[C1['in_max'] == 0]; d4 = coh4['post30_day_first']
ADDL['first_echo_day_v5'] = dict(median=float(q1day.median()), q1=float(q1day.quantile(0.25)), q3=float(q1day.quantile(0.75)))
ADDL['first_echo_day_v4'] = dict(median=float(d4.median()), q1=float(d4.quantile(0.25)), q3=float(d4.quantile(0.75)))
ADDL['uptake_v5'] = rate(len(Q1), len(U30)); ADDL['uptake_v4'] = rate(len(coh4), len(u4))
# (d) v4 value of S3 (aneurysm on any echocardiogram within 30 days), for context
ADDL['S3_v4'] = rate(int((coh4['post30_cat_max'] >= 2).sum()), len(coh4))
# (e) wording of plan 4(b): segment term and 扩张|瘤 anywhere in the conclusion (implemented) vs within the same clause (stricter reading)
clause_hit = conc.map(lambda t: any(NONPROX.search(c) and re.search(r'扩张|瘤', c) for c in re.split(r'[，,。；;]', t)))
ov_clause = EW['num_cat'].notna() & (EW['num_cat'] < 2) & ~text_num & (EW['wording_cat'] >= 2) & clause_hit
EWc = EW[['report_key', 'research_patient_id', 'exam_date', 'phase']].copy()
EWc['cat_override_clause'] = np.select([ov_clause, EW['num_cat'].notna(), fallback_t3, fallback_t4], [EW['wording_cat'], EW['num_cat'], EW['aha_best'], EW['wording_cat']], np.nan)
ch_c, _ = build('cat_override_clause', EWc)
v_c = ch_c[ch_c['inmax'].notna()]; q_c = v_c[(v_c['inmax'] == 0) & (v_c['dis'] <= END30) & v_c['q1_cat'].notna()]
ADDL['override_reports_anywhere_vs_clause'] = [int(override.sum()), int(ov_clause.sum())]
ADDL['P1_override_clause_reading'] = dict(rate(int((q_c['q1_cat'] >= 2).sum()), len(q_c)), predictive_P=predictive_p(int((q_c['q1_cat'] >= 2).sum()), len(q_c), *V4['P1']))
# (f) how close the S2 reading is to the threshold: smallest n at which 0 events gives predictive P < 0.05, and P under rule (b)
ADDL['S2_zero_events_min_n_for_P_below_0.05'] = next(n for n in range(1, 5000) if predictive_p(0, n, *V4['S2'], digits=None) < 0.05)
ch_b, _ = build('cat_wording_v4')
q_b = ch_b[(ch_b['inmax'] == 0) & (ch_b['dis'] <= END30) & ch_b['q1_cat'].notna()]
k_b = int((q_b['q1_cat'] >= 3).sum())
ADDL['S2_rule_b'] = dict(rate(k_b, len(q_b)), predictive_P=predictive_p(k_b, len(q_b), *V4['S2']))
for k, v in ADDL.items():
    print(k, v)

## 11. Figure V5-1
A, the pre-specified endpoints in the locked 2018–2024 cohort (v4), the 2025–2026 validation cohort and both periods pooled. B, the primary endpoint in the validation cohort by the source of the coronary category, against the locked v4 estimate. P values are given in the figure legend file.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'font.size': 11,
                     'axes.labelsize': 12, 'axes.labelweight': 'bold', 'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.grid': False, 'savefig.dpi': 300, 'pdf.fonttype': 42})
BLACK, DGREY, MGREY, LGREY = '#000000', '#4d4d4d', '#8c8c8c', '#c8c8c8'


def savefig(fig, name):
    fig.savefig(os.path.join(OUT, f'{name}.png'), bbox_inches='tight', facecolor='white')
    fig.savefig(os.path.join(OUT, f'{name}.pdf'), bbox_inches='tight', facecolor='white', metadata={'CreationDate': None})
    plt.show(); plt.close(fig)


def letters(fig, pairs, dy=0.012):
    """Panel letters at the left edge of each panel's y-axis (tick labels + title), level with the top of the axes."""
    fig.canvas.draw(); r_ = fig.canvas.get_renderer(); inv = fig.transFigure.inverted()
    for ax, L in pairs:
        x0 = inv.transform((ax.yaxis.get_tightbbox(r_).x0, 0))[0]; y1 = inv.transform((0, ax.get_window_extent(r_).y1))[1]
        fig.text(x0, y1 + dy, L, fontsize=15, fontweight='bold', ha='left', va='bottom')


C = CMP.set_index('key')
EPS = [('P1', 'Aneurysm at the first\necho 1–30 d (P1)'), ('S1', 'Any involvement at the\nfirst echo 1–30 d (S1)'),
       ('S2', 'Medium or larger at the\nfirst echo 1–30 d (S2)'), ('S4_aneurysm', 'Aneurysm at 31–90 d after\nnormal 1–30-d echoes (S4)')]
SER = [('v4', '2018–2024, locked v4', 'o', 'white', BLACK), ('v5', '2025–2026, validation', 's', BLACK, BLACK), ('pooled', 'Pooled 2018–2026', 'D', MGREY, MGREY)]
XA, XB = 3.6, 3.0
fig, axes = plt.subplots(1, 2, figsize=(16, 5.8), gridspec_kw={'width_ratios': [1.25, 1], 'wspace': 0.95})
ax = axes[0]; base = np.arange(len(EPS))[::-1].astype(float)
for b, (k, lab) in zip(base, EPS):
    r = C.loc[k]
    for j, (s, slab, mk, fc, ec) in enumerate(SER):
        y = b + 0.27 - 0.27 * j
        v, lo, hi, e, n = [r[f'{s}_{c}'] for c in ['pct', 'ci_low', 'ci_high', 'events', 'n']]
        ax.errorbar(v, y, xerr=[[v - lo], [hi - v]], fmt=mk, color=ec, markerfacecolor=fc, markeredgecolor=ec, ecolor=ec, elinewidth=1, capsize=2.5, markersize=6, clip_on=False, zorder=3)
        ax.text(XA + 0.06, y, f'{v:.2f} ({lo:.2f}–{hi:.2f}); {int(e)}/{int(n):,}', va='center', fontsize=8.5, clip_on=False)
for sep in base[:-1] - 0.5:
    ax.axhline(sep, color=LGREY, lw=0.6)
ax.set_yticks(base); ax.set_yticklabels([lab for _, lab in EPS], fontsize=10)
ax.set_xlim(0, XA); ax.set_ylim(base.min() - 0.55, base.max() + 0.55); ax.set_xlabel('Children with the finding (%), 95% CI')
ax.legend(handles=[Line2D([], [], marker=mk, color=ec, markerfacecolor=fc, markeredgecolor=ec, linestyle='none', markersize=6, label=slab) for s, slab, mk, fc, ec in SER],
          frameon=False, fontsize=9, loc='lower right')
ax = axes[1]
SLAB = ['Primary: combined rule', '(a) Without the\nwording override', '(b) Report categories\nof v4 (wording model)']
yy = np.arange(len(SENS))[::-1].astype(float)
v4p, v4l, v4h = C.loc['P1', 'v4_pct'], C.loc['P1', 'v4_ci_low'], C.loc['P1', 'v4_ci_high']
ax.axvspan(v4l, v4h, color=LGREY, alpha=0.45, lw=0)
ax.axvline(v4p, color=DGREY, lw=1, ls='--')
for yi, (_, r) in zip(yy, SENS.iterrows()):
    mk, fc = ('s', BLACK) if yi == yy[0] else ('o', 'white')
    ax.errorbar(r['pct'], yi, xerr=[[r['pct'] - r['ci_low']], [r['ci_high'] - r['pct']]], fmt=mk, color=BLACK, markerfacecolor=fc, ecolor=BLACK, elinewidth=1, capsize=2.5, markersize=6, clip_on=False, zorder=3)
    ax.text(XB + 0.06, yi, f"{r['pct']:.2f} ({r['ci_low']:.2f}–{r['ci_high']:.2f}); {int(r['events'])}/{int(r['n']):,}", va='center', fontsize=8.5, clip_on=False)
ax.set_yticks(yy); ax.set_yticklabels(SLAB, fontsize=10); ax.set_xlim(0, XB); ax.set_ylim(-0.6, len(SENS) - 0.4)
ax.set_xlabel('Aneurysm at the first echo 1–30 d (%), 95% CI')
ax.legend(handles=[Line2D([], [], color=DGREY, lw=1, ls='--', label=f'Locked v4 estimate, 2018–2024 ({v4p:.2f}%)'),
                   Patch(facecolor=LGREY, alpha=0.45, lw=0, label=f'95% CI of the v4 estimate ({v4l:.2f}–{v4h:.2f})')],
          frameon=False, fontsize=9, loc='lower left', bbox_to_anchor=(0.0, 1.0))
letters(fig, [(axes[0], 'A'), (axes[1], 'B')])
savefig(fig, 'FigureV5_1_temporal_validation')

## 12. Results lock, key numbers, figure legends and the kit v1.1 manifest

In [ ]:
def fr(d):
    return f"{d['events']}/{d['n']:,} = {d['pct']:.2f}% (95% CI {d['ci_low']:.2f}–{d['ci_high']:.2f})"


def frow(r, pre):
    return f"{int(r[pre + '_events'])}/{int(r[pre + '_n']):,} = {r[pre + '_pct']:.2f}% (95% CI {r[pre + '_ci_low']:.2f}–{r[pre + '_ci_high']:.2f})"


def fp(p):
    return 'P < 0.001' if p < 0.001 else f'P = {p:.3f}' if p < 0.1 else f'P = {p:.2f}'


def cmp_lines(k, label):
    r = C.loc[k]
    return [f"- {label}: {frow(r, 'v5')} vs v4 {frow(r, 'v4')}.",
            f"  Difference {r['diff_pp']:+.2f} percentage points (Newcombe 95% CI {r['diff_ci_low']:+.2f} to {r['diff_ci_high']:+.2f}); expected events under v4 {r['expected_v5_events_under_v4']:.1f}; "
            f"predictive compatibility {fp(r['predictive_P'])} -> {r['reading']}"
            + ('' if r['reading'] == 'replicated' else (' (fewer events than predicted)' if r['v5_events'] < r['expected_v5_events_under_v4'] else ' (more events than predicted)')) + '.',
            f"  Pooled 2018–2026: {int(r['pooled_events'])}/{int(r['pooled_n']):,} = {r['pooled_pct']:.2f}% (95% CI {r['pooled_ci_low']:.2f}–{r['pooled_ci_high']:.2f})."]


inm = V['inmax'].value_counts().reindex([0, 1, 2, 3, 4]).fillna(0).astype(int)
yrs = CAND.loc[V.index, 'index_year'].value_counts().sort_index()
manifest_rows = []
for f in sorted(glob.glob(os.path.join(ADD, 'KD_*.parquet'))) + sorted(glob.glob(os.path.join(ADD, 'KD_*.csv'))):
    df_ = pd.read_parquet(f) if f.endswith('.parquet') else None
    manifest_rows.append(dict(file=os.path.basename(f), rows=len(df_) if df_ is not None else len(pd.read_csv(f, encoding='utf-8-sig')),
                              columns=df_.shape[1] if df_ is not None else pd.read_csv(f, encoding='utf-8-sig', nrows=0).shape[1], sha256=sha256(f)))
MAN = pd.DataFrame(manifest_rows)
L = []
L.append('RESULTS LOCK — v5 (2026-10-14) — temporal validation of the KD surveillance-echo results, Wuhan Children\'s Hospital')
L.append('Computed by KD_echo_surveillance_v5_validation.ipynb. Every number is computed. Locked v4/v4.1 results are unchanged.')
L.append('')
L.append('V0. Plan, data and integrity')
L.append(f'- Plan: V5_VALIDATION_PLAN.txt, SHA-256 {PLAN_SHA256_LOCKED} (locked 2026-10-14T10:47:36Z, before any validation outcome was computed).')
L.append(f'- Locked references: Results_locked_v4.txt SHA-256 {sha256(LOCK4_PATH)}; Results_locked_v4_1.txt SHA-256 {REF41_SHA256}.')
L.append('- Structured export: ' + (f'{os.path.basename(EXPORT)}, SHA-256 {EXPORT_SHA256}' if EXPORT else 'not present in this run; frozen measurement table used') + '.')
for _, m in MAN[MAN['file'].str.endswith('.parquet')].iterrows():
    L.append(f"- Kit v1.1 table {m['file']}: {m['rows']:,} rows, SHA-256 {m['sha256']}.")
L.append(f'- Export window {EXPORT_START.date()} to {DATA_END.date()} (DATA_END); {len(MEAS):,} reports of {MEAS["research_patient_id"].nunique():,} children; '
         f'{len(COMB):,} frozen reports in the window, all matched to the structured table by research patient ID and exam date.')
src_ = COMB['combined_src'].value_counts()
L.append('- Source of the per-report category: ' + '; '.join(f'{k} {int(v):,}' for k, v in src_.items()) + '.')
L.append('- v4 reference values recomputed from cohort_C1_primary_v4 and found in Results_locked_v4.txt: ' + '; '.join(f'{k} {a}/{n:,}' for k, (a, n) in V4.items()) + '.')
L.append('')
L.append('V1. Validation cohort (V)')
L.append(f'- Index admissions with the first in-hospital echocardiogram on or after {EXPORT_START.date()}: {len(CAND):,}; with a classifiable in-hospital report (V): {len(V):,} '
         f"({', '.join(f'{int(y)}: {int(c)}' for y, c in yrs.items())}).")
L.append('- In-hospital maximum: ' + '; '.join(f'{CATNAME[i]} {inm[i]:,}' for i in range(5)) + '.')
L.append(f"- Uncomplicated (in-hospital maximum none): {len(U):,}; complete 30-day follow-up: {len(U30):,}; first classifiable echocardiogram 1–30 d after discharge: {len(Q1):,} "
         f"({ADDL['uptake_v5']['pct']:.1f}%; v4 {ADDL['uptake_v4']['events']:,}/{ADDL['uptake_v4']['n']:,} = {ADDL['uptake_v4']['pct']:.1f}%); first echo on day "
         f"{ADDL['first_echo_day_v5']['median']:.0f} after discharge [IQR {ADDL['first_echo_day_v5']['q1']:.0f}–{ADDL['first_echo_day_v5']['q3']:.0f}] (v4 {ADDL['first_echo_day_v4']['median']:.0f} [{ADDL['first_echo_day_v4']['q1']:.0f}–{ADDL['first_echo_day_v4']['q3']:.0f}]).")
L.append('- The preliminary pre-lock counts in plan section 3 (812 uncomplicated; 546 with a first echocardiogram 1–30 d; 311 for 31–90 d) came from a stricter draft screening rule '
         '(every in-hospital report matched to the export and normal by structured Z, text Z and wording, including "upper limit of normal" wording, and any post-discharge report counted); '
         'the final cohort follows plan sections 4–5.')
L.append('')
L.append('V2. Primary endpoint P1 (aneurysm at the first classifiable echocardiogram 1–30 d after discharge, uncomplicated course)')
L += cmp_lines('P1', 'P1')
L.append('')
L.append('V3. Secondary endpoints')
L += cmp_lines('S1', 'S1 any involvement at the same echocardiogram')
L += cmp_lines('S2', 'S2 medium or larger at the same echocardiogram')
L.append(f"- S3 aneurysm on any echocardiogram within 30 d: {fr(S3)} (v4, for context: {fr(ADDL['S3_v4'])}).")
L += cmp_lines('S4_aneurysm', 'S4 aneurysm at 31–90 d after normal 1–30-d echocardiograms')
r_ = C.loc['S4_involvement']
L.append(f"- S4 any involvement at 31–90 d: {frow(r_, 'v5')} (v4 {frow(r_, 'v4')}; comparison not pre-specified: predictive {fp(r_['predictive_P'])}).")
L.append(f"- S5 91–365 d after normal findings in all earlier windows (complete 365-day follow-up): aneurysm {fr(S5['aneurysm'])}; any involvement {fr(S5['involvement'])}.")
L.append(f"- S6 persistence of the P1 events (next classifiable echocardiogram ≤90 d): confirmed {S6['confirmed']}, not confirmed {S6['not confirmed']}, not re-imaged {S6['not re-imaged']}; "
         f"size at detection {S6_size}; days to the next echocardiogram median {S6_gap.median():.1f} (range {S6_gap.min():.0f}–{S6_gap.max():.0f}).")
L.append(f"- S7 risk-factor strategy (echo only if ≥1 of age <12 months, IVIG resistance, in-hospital dilation only, discharge CRP >20 mg/L or WBC >12 ×10⁹/L), children discharged with "
         f"in-hospital maximum ≤ dilation only and a first echo ≤30 d (complete follow-up): n = {S7['children']:,}, aneurysms {S7['aneurysms']}; echoes {S7['echoes_if_ge1']:,} "
         f"(avoided {S7['echoes_avoided_pct']:.1f}%); detected {S7['detected']}, missed {S7['missed']}"
         + (f" {S7['missed_sizes']}" if S7['missed'] else '') + (f"; sensitivity {fr(S7['sensitivity'])}" if S7['sensitivity'] else '')
         + f"; aneurysms by risk-factor count: 0 factors {S7['by_count'].get(0, '0/0')}, 1 factor {S7['by_count'].get(1, '0/0')}, ≥2 factors {S7['by_count'].get(2, '0/0')}.")
for _, r in S8.iterrows():
    L.append(f"- S8 cohort C1, {r['scope']}: W3 aneurysm {r['W3']} (95% CI {r['W3_ci']}), any involvement {r['W3_involvement']}; W4 aneurysm {r['W4']} (95% CI {r['W4_ci']}), any involvement {r['W4_involvement']}.")
L.append('  The locked-scope row reproduces the v4 values (W3 0/609, involvement 2/609; W4 0/221, involvement 0/221).')
L.append('')
L.append('V4. Sensitivity analyses of P1 (source of the coronary category)')
for _, r in SENS.iterrows():
    L.append(f"- {r['analysis']}: {int(r['events'])}/{int(r['n']):,} = {r['pct']:.2f}% (95% CI {r['ci_low']:.2f}–{r['ci_high']:.2f}); predictive compatibility with v4 {fp(r['predictive_P'])}.")
L.append('')
L.append('V5. Additional analyses (not pre-specified; descriptive)')
L.append(f"- Confirmed new aneurysms (Z ≥ 2.5 again on the next echo ≤90 d; definition (a) of v4.1 module M2b): v5 {fr(ADDL['confirmed_yield_v5'])}; v4.1 {fr(ADDL['confirmed_yield_v41'])}.")
L.append(f"- Confirmed among re-imaged: v5 {fr(ADDL['confirmed_among_reimaged_v5'])}; v4.1 {fr(ADDL['confirmed_among_reimaged_v41'])}; Fisher exact {fp(ADDL['fisher_P_confirmed_among_reimaged_posthoc'])} (post hoc). "
         f"Not re-imaged ≤90 d: v5 {ADDL['not_reimaged_v5']}, v4.1 {ADDL['not_reimaged_v41']}.")
L.append(f"- P1 events one by one (TableV5_4_P1_events.csv): days after discharge {', '.join(str(x) for x in DET['day_after_discharge'])}; structured Z at detection "
         f"{', '.join(f'{x:.2f}' for x in DET['structured_Z'])}; maximal in-hospital structured Z {', '.join(f'{x:.2f}' for x in DET['inhospital_max_structured_Z'])}; "
         f"source {', '.join(DET['source'])}; last category {', '.join(c if c else 'NA' for c in DET['last_echo_category'])} at day "
         f"{', '.join(str(int(x)) if pd.notna(x) else 'NA' for x in DET['last_echo_day_after_discharge'])}.")
L.append('')
L.append('V6. Deviations from the plan')
L.append('- None in the pre-specified analyses. Additions (labelled): the S4 any-involvement comparison with v4, section V5, and the expected event counts under v4.')
L.append(f"- Post hoc check of the wording of plan 4(b): the override was implemented with the segment term and 扩张|瘤 anywhere in the conclusion ({ADDL['override_reports_anywhere_vs_clause'][0]} reports); "
         f"requiring both in the same clause gives {ADDL['override_reports_anywhere_vs_clause'][1]} reports and P1 {fr(ADDL['P1_override_clause_reading'])}, predictive {fp(ADDL['P1_override_clause_reading']['predictive_P'])}.")
L.append('- Implementation notes: the export carries no report key, so structured values were linked to the frozen reports by research patient ID and exam date (day level), in line with the same-day maximum rule; '
         'six exact duplicate rows of the frozen report table were dropped, as stated in the plan.')
L.append('')
L.append('V7. Reading and caveats')
L.append('- "Replicated" means no detectable incompatibility with v4 (beta-binomial predictive P ≥ 0.05); it is not proof of equality. Event counts in V are small.')
L.append(f"- The S2 reading sits at the threshold: with 0 events the predictive P falls below 0.05 only from n = {ADDL['S2_zero_events_min_n_for_P_below_0.05']}; "
         f"with the v4 report categories (rule b) S2 is {fr(ADDL['S2_rule_b'])}, predictive {fp(ADDL['S2_rule_b']['predictive_P'])}. The direction is fewer medium or larger aneurysms than v4 predicts.")
L.append('- Temporal, single-center validation with the same laboratory Z equations; the structured table covers proximal segments only (non-proximal findings rely on text numbers or the wording override).')
L.append('- Children without a classifiable echocardiogram 1–30 d after discharge are not in the P1 denominator; 2026 admissions have short follow-up beyond 90 d.')
RES = '\n'.join(L) + '\n'
p_res = os.path.join(OUT, 'Results_locked_v5.txt')
open(p_res, 'w', encoding='utf-8').write(RES)
print(RES)


def jsonable(o):
    if isinstance(o, dict):
        return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jsonable(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if pd.isna(o) else round(float(o), 4)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o


KEY = dict(plan_sha256=PLAN_SHA256_LOCKED, export_sha256=EXPORT_SHA256, export_window=[str(EXPORT_START.date()), str(DATA_END.date())],
           addendum={m['file']: dict(rows=int(m['rows']), sha256=m['sha256']) for _, m in MAN.iterrows()},
           v4_reference={k: dict(events=a, n=n) for k, (a, n) in V4.items()},
           cohort=dict(candidates=len(CAND), V=len(V), by_year={int(y): int(c) for y, c in yrs.items()}, inhospital_max={CATNAME[i]: int(inm[i]) for i in range(5)},
                       uncomplicated=len(U), complete_30d=len(U30), Q1=len(Q1), W2=len(W2), W3=len(W3)),
           comparisons={r['key']: {c: r[c] for c in CMP.columns if c not in ('key',)} for _, r in CMP.iterrows()},
           S3=S3, S5=S5, S6=dict(S6, size=S6_size, days_to_next=list(S6_gap)), S7=S7, S8=S8.to_dict('records'), sensitivity=SENS.to_dict('records'), additional=ADDL)
p_key = os.path.join(OUT, 'key_numbers_v5.json')
json.dump(jsonable(KEY), open(p_key, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
LEG = f"""FIGURES AND TABLES — v5 temporal validation (output_kd_echo_surveillance_v5)

FigureV5_1_temporal_validation (.png/.pdf)
Temporal validation of the yield of post-discharge echocardiography after an uncomplicated in-hospital course. A, proportion of children with each finding in the locked 2018–2024 cohort (v4; open circles), the 2025–2026 validation cohort (filled squares) and both periods pooled (grey diamonds); bars are Wilson 95% CIs; labels give the percentage (95% CI) and events/children. Predictive compatibility of the validation count with v4 (two-sided beta-binomial P): aneurysm at the first echo 1–30 d {fp(C.loc['P1', 'predictive_P'])}; any involvement {fp(C.loc['S1', 'predictive_P'])}; medium or larger {fp(C.loc['S2', 'predictive_P'])}; aneurysm at 31–90 d {fp(C.loc['S4_aneurysm', 'predictive_P'])}. B, aneurysm at the first echo 1–30 d in the validation cohort by the source of the coronary category: combined rule (primary), without the wording override, and the v4 report categories; dashed line and grey band, locked v4 estimate and its 95% CI. Predictive P: {'; '.join(f"{a.split(':')[0]} {fp(p)}" for a, p in zip(SENS['analysis'], SENS['predictive_P']))}.

TableV5_1_validation_vs_v4.csv — pre-specified comparisons with v4 (Newcombe difference, beta-binomial predictive P, pooled estimate); the S4 any-involvement comparison is not pre-specified (column prespecified_comparison).
TableV5_2_P1_sensitivity.csv — P1 by the source of the coronary category.
TableV5_3_extended_followup_C1.csv — S8, late windows of cohort C1 in the locked scope and extended to {DATA_END.date()}.
TableV5_4_P1_events.csv — the P1 events one by one (research IDs only; additional, descriptive).
Results_locked_v5.txt — all numbers with the plan and data hashes; key_numbers_v5.json — the same in machine-readable form.
"""
open(os.path.join(OUT, 'Figures_and_tables_index_v5.txt'), 'w', encoding='utf-8').write(LEG)
MAN['description'] = MAN['file'].map(lambda f: 'Structured coronary measurements, one row per exported report (2025-03-08 to 2026-09-23)' if 'measure' in f
                                     else 'Per-report combined coronary category for frozen reports in the export window (plan section 4)')
MAN.to_csv(os.path.join(ADD, 'MANIFEST_v1_1.csv'), index=False)
json.dump(dict(run_utc=datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'), python=sys.version.split()[0], pandas=pd.__version__, numpy=np.__version__),
          open(os.path.join(OUT, 'run_info_v5.json'), 'w'), indent=1)   # run time and versions (not part of the checksums)
SUMS = [(os.path.relpath(f, BASE), sha256(f)) for f in sorted(glob.glob(os.path.join(OUT, '*'))) if os.path.basename(f) not in ('CHECKSUMS_v5.sha256', 'run_info_v5.json')]
open(os.path.join(OUT, 'CHECKSUMS_v5.sha256'), 'w', encoding='utf-8').write(''.join(f'{h}  {p}\n' for p, h in SUMS))
print('Results_locked_v5.txt SHA-256', sha256(p_res))
print(MAN.to_string(index=False))